# 01 - Tokenizer

In [1]:
import json
from pathlib import Path
import numpy as np

root = Path("../data/processed/shapes_v1")
manifest = json.loads((root / "manifest.json").read_text())
splits = ["train", "val", "test_iid", "test_comp"]
data = {s: np.load(root / f"{s}.npz") for s in splits}

print(manifest["counts"])
d = data["train"]
for k in d.files:
    print(f"{k:10s} {d[k].shape}  {d[k].dtype}")
print(d["captions"][:3])

{'train': 200000, 'val': 5000, 'test_iid': 5000, 'test_comp': 10000}
images     (200000, 16, 16)  uint8
n_obj      (200000,)  uint8
color      (200000, 2)  int8
shape      (200000, 2)  int8
size       (200000, 2)  int8
pos        (200000,)  int8
rel        (200000,)  int8
jitter     (200000, 2, 2)  int8
captions   (200000,)  <U56
['a large cyan triangle above a small white square'
 'a small orange cross right of a small orange square'
 'a large white cross above a small cyan circle']


In [2]:
manifest["text_vocab"]

['a',
 'above',
 'at',
 'below',
 'blue',
 'bottom',
 'center',
 'circle',
 'cross',
 'cyan',
 'green',
 'in',
 'large',
 'left',
 'of',
 'on',
 'orange',
 'purple',
 'red',
 'right',
 'small',
 'square',
 'the',
 'top',
 'triangle',
 'white',
 'yellow']

In [ ]:
PAD, UNK = "<pad>", "<unk>"

class TextTokenizer:
    def __init__(self, words):
        self.tokens = [PAD, UNK] + sorted(set(words))
        self.token_to_id = {token:i for i, token in enumerate(self.tokens)}
        self.id_to_token = {i:token for i, token in enumerate(self.tokens)}
    
    @property
    def vocab_size(self) -> int:
        return (
            len(self.tokens)
        )

    @property
    def pad_id(self) -> int:
        return (
            self.token_to_id[PAD]
        )

    def encode(self, caption: str, max_len: int) -> list[int]:
        # 1) Parte el caption en palabras con caption.split().
        words = caption.split()
        # 2) Si hay más de max_len palabras: lanza ValueError.
        if len(words) > max_len:
            raise ValueError(f"Caption of {len(words)} words is above max_len={max_len}")

        # 3) Convierte cada palabra a su id; si no existe, usa el id de UNK.
        ids = []
        for word in words:
            if word in self.tokens:
                ids.append(self.token_to_id[word])
            else:
                ids.append(self.token_to_id[UNK])
                
        # 4) Rellena con pad_id hasta llegar a max_len.
        ids += [self.pad_id] * (max_len - len(ids))
        return  ids

    
    def decode(self, ids) -> str:
        # Convierte cada id a su palabra, ignora los ids de PAD y une con espacios.
        words = [
            self.id_to_token[i] for i in ids if i != self.pad_id        
        ]
        return " ".join(words)




In [36]:
tok = TextTokenizer(manifest["text_vocab"])
tok.vocab_size
tok.pad_id

max_len = max(
    len(c.split())
    for s in splits
    for c in data[s]["captions"]
)
print(max_len)

ids = tok.encode(data["train"]["captions"][0], max_len)
print(ids)
print(tok.decode(ids))

10
[2, 14, 11, 26, 3, 2, 22, 27, 23, 0]
a large cyan triangle above a small white square


In [34]:
tok = TextTokenizer(manifest["text_vocab"])
ids = tok.encode("a red circle", 6)
print(ids)                # debe terminar en tres ceros (relleno)
print(tok.decode(ids))    # "a red circle"
print(tok.encode("a purple dragon", 6))   # "dragon" debe salir como 1 (<unk>)

[2, 20, 9, 0, 0, 0]
a red circle
[2, 19, 1, 0, 0, 0]
